# Exploración inicial del entorno de exploración de Flappy Bird

**Descripción**: En este notebook importamos y probamos el entorno `flappy-bird-gymnasium` para tomar decisiones con respecto al diseño de los procesos de entrenamiento aproximados.

## 1. Preparación del entorno

A continuación se realizan todas las importaciones de librería de Python necesarias para este estudio.

In [ ]:
!git clone https://github.com/rafelsalgueiro/GallegoSalgueiroVera.git

In [1]:
!pip install -q flappy-bird-gymnasium gymnasium numpy matplotlib seaborn pandas

In [11]:
import sys
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym
import flappy_bird_gymnasium


project_root = os.path.abspath('GallegoSalgueiroVera/Entornos_Complejos')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

try:
    print("Entorno cargado correctamente.")

except ImportError as e:
    print(f"Error cargando el entorno: {e}")
    sys.exit(1)

Entorno cargado correctamente.


Y definimos semillas de números aleatorios para reproducibilidad.

In [12]:
SEED = 31415
np.random.seed(SEED)

## 2. Exploración del entorno

`flappy-bird-gimnasium` ofree dos modos de observación:
- **use_lidar=False**: doce observaciones:
    - Posición horizontal de la última tubería.
    - Posición vertical de la última tubería superior.
    - Posición vertical de la última tubería inferior.
    - Posición horizontal de la próxima tubería.
    - Posición vertical de la próxima tubería superior.
    - Posición vertical de la próxima tubería inferior.
    - Posición horizontal de la siguiente a la próxima tubería.
    - Posición vertical de la siguiente a la próxima tubería.
    - Posición vertical de la siguiente a la próxima tubería.
    - Posición vertical del personaje.
    - Velocidad vertical del personaje.
    - Rotación del personaje.
- **use_lidar=True**: vector de 180 características con distancias medidas por sensores.

In [13]:
# Entorno con lidar=false
env_no_lidar = gym.make("FlappyBird-v0", render_mode=None, use_lidar=False)
print("=== NO-LIDAR MODE ===")
print(f"  Obs space:  {env_no_lidar.observation_space}")
print(f"  Obs shape:  {env_no_lidar.observation_space.shape}")
print(f"  Obs dtype:  {env_no_lidar.observation_space.dtype}")
print(f"  Action space: {env_no_lidar.action_space}  (n={env_no_lidar.action_space.n})")

# Entorno con lidar=true
env_lidar = gym.make("FlappyBird-v0", render_mode=None, use_lidar=True)
print("\n=== LIDAR MODE ===")
print(f"  Obs space:  {env_lidar.observation_space}")
print(f"  Obs shape:  {env_lidar.observation_space.shape}")
print(f"  Obs dtype:  {env_lidar.observation_space.dtype}")
print(f"  Obs low:    {env_lidar.observation_space.low}")
print(f"  Obs high:   {env_lidar.observation_space.high}")

# Reset para obtener una muestra de observación y el diccionario de información
obs_no_lidar, info_nl = env_no_lidar.reset(seed=SEED)
obs_lidar, info_l = env_lidar.reset(seed=SEED)

print(f"\n--- Sample pixel obs shape: {obs_no_lidar.shape}, min={obs_no_lidar.min()}, max={obs_no_lidar.max()}")
print(f"--- Sample LIDAR obs: {obs_lidar}")
print(f"--- Info dict keys: {list(info_l.keys())}")
print(f"--- Info contents:  {info_l}")

env_no_lidar.close()
env_lidar.close()

=== NO-LIDAR MODE ===
  Obs space:  Box(-1.0, 1.0, (12,), float64)
  Obs shape:  (12,)
  Obs dtype:  float64
  Action space: Discrete(2)  (n=2)

=== LIDAR MODE ===
  Obs space:  Box(0.0, 1.0, (180,), float64)
  Obs shape:  (180,)
  Obs dtype:  float64
  Obs low:    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  Obs high:   [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 

/home/juandi/Desktop/Master/EML/GallegoSalgueiroVera/.venv/lib/python3.12/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


Tras ejecutar nuestra celda de exploración confirmamos la forma en la que se percibe el entorno:

1. Modo No-LIDAR
    - El espacio se representa como: `Box(-1.0, 1.0, (12,), float64)`
    - Los valores están normalizados entre `-1.0` y `1.0`.
2. Modo LIDAR
    - El espacio se representa como `Box(0.0, 1.0, (180,), float64)`
    - El agente emite 180 rayos láser virtuales a su alrededor.
    - En la mayoría de observaciones vemos `1.0`, entendiéndose como: no hay obstáculos en el alcance máximo fijado.
    - El resto de valores representan una distancia detectable por el personaje, siendo `0.0` un contacto directo.

Además tenemos una visión del espacio de acción y del diccionario de información:
    - Solo tenemos dos acciones: caer o saltar.
    - La clave `score` es la utilizada para mantener la puntuación real del juego.

Decidimos proseguir con el espacio No-Lidar dada su sencillez.

## 3. Recompensas

Según la documentación, el entorno utiliza las siguientes recompensas:
    - +0.1 por sobrevivir cada frame.
    - +1.0 por atravesar una tubería.
    - -1.0 por morir.
    - -0.5 por tocar la  parte superior de la pantalla.
Aunque esto es suficiente para que el algoritmo converja eventualmente, podemos probar diferentes estrategias para acelerar el aprendizaje.

Queremos incentivar al pájaro para que no solo sobreviva, sino que se alinee con el hueco del próximo tubo. Para ello, aplicaremos una penalización suave y proporcional a la distancia entre la altura del pájaro y el centro del hueco.
$$R:=R-\alpha\cdot |y_{personaje}-y_{centro}|$$


In [14]:
class FlappyBirdRewardWrapper(gym.Wrapper):
    """
    Custom Wrapper para modificar las recompensas del entorno FlappyBird-v0.
    Incentiva al agente a mantenerse en el centro del hueco de los próximos tubos.
    """
    def __init__(self, env, alpha=0.5):
        super().__init__(env)
        self.alpha = alpha
        
    def step(self, action):
        # Ejecutar la acción en el entorno original
        obs, reward, terminated, truncated, info = self.env.step(action)
        
        # Aplicar Reward Shaping solo si el agente sigue vivo
        if not terminated:
            next_top_pipe_y = obs[4]
            next_bottom_pipe_y = obs[5]
            player_y = obs[9]
            
            gap_center = (next_top_pipe_y + next_bottom_pipe_y) / 2.0
            distance_to_center = abs(player_y - gap_center)
            shaping_penalty = -self.alpha * distance_to_center
            reward += shaping_penalty
            
        return obs, reward, terminated, truncated, info

# Creamos el entorno base y lo envolvemos
env_base = gym.make("FlappyBird-v0", use_lidar=False)
env_shaped = FlappyBirdRewardWrapper(env_base, alpha=0.5)

obs_shaped, _ = env_shaped.reset(seed=SEED)
next_obs, shaped_reward, done, _, _ = env_shaped.step(1) # Acción 1: Saltar

print(f"Recompensa modificada tras el primer salto: {shaped_reward:.4f}")

env_shaped.close()

Recompensa modificada tras el primer salto: 0.0795


## 4. Visualización del agente

Antes de decidir ninguna estrategia de aprendizaje resulta interesante probar la capacidad de renderización del juego para establecer una línea base visual. Con la celda inferior podemos abrir una ventana con el juego.

In [16]:
# Para visualizar, necesitamos crear una nueva instancia con render_mode="human"
env_vis = gym.make("FlappyBird-v0", use_lidar=False, render_mode="human")

obs, info = env_vis.reset(seed=SEED)
score = 0

for step in range(500):
    action = env_vis.action_space.sample()

    obs, reward, terminated, truncated, info = env_vis.step(action)
    
    # Ralentizamos a 30 FPS para que sea visible
    time.sleep(1 / 30.0)
    
    if terminated or truncated:
        score = info.get('score', 0)
        print(f"Agente colisionó en el timestep {step}. Puntuación final: {score}")
        break

env_vis.close()

Agente colisionó en el timestep 49. Puntuación final: 0


## 5. Conclusiones

Hemos completado con éxito la exploración del entorno y tomado las siguientes decisiones:
- Estado: decidimos que el modo no-LIDAR es el más adecuado pues nos proporciona un vector simple y suficiente para un entrenamiento eficaz.
- Recompensa: implementamos un *reward shaping* para guiar al agente hacia el centro de los tubos. Evaluaremos el impacto de este factor en diferentes entrenamientos.
- Infraestructura: guardaremos el rendimiento medio de los agentes en progresivas iteraciones de entrenamiento para evaluar capacidad y velocidad de aprendizaje.

A partir de este estudio preliminar dividimos el flujo de trabajo en dos notebooks:
 - sarsa_semi_gradient.ipynb,
 - deep_q_learning.ipynb,
probando dos diferentes métodos aproximados.